# Parallelization

Run independent subtasks at the same time and merge their results.



## Core ideas



- Independence: parallel work is safe only when subtasks do not require each other's outputs.
- Fan-out and fan-in: distribute work, then merge it with a deterministic or review-based reducer.
- Diversity: parallel agents or prompts can explore different perspectives or candidates.
- Resource tradeoff: parallelism lowers latency but can increase cost and coordination complexity.
- Aggregation: merging is often the hardest part and should include conflict handling.


## Common failure modes

- Parallelizing dependent steps.
- Ignoring merge conflicts.
- Spending more on parallel branches than the task value justifies.


## Framework implementation

Use parallel LangGraph branches when the work is independent. The list reducer makes concurrent updates explicit and deterministic at fan-in.


In [ ]:
# Import the dependencies used by this example.
import operator
import os
from typing import Annotated, TypedDict
from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph

# Define the data shape and small operations before running them.
class State(TypedDict):
    topic: str
    findings: Annotated[list[str], operator.add]

if not os.getenv("OPENAI_API_KEY"):
    print("Skipped: set OPENAI_API_KEY to run parallel model calls.")
else:
    # Configure the framework object; this line prepares it but may not execute it yet.
    model = init_chat_model("openai:gpt-5.6-sol", temperature=0)

    def analyze(angle: str):
        def node(state: State):
            # Execute the configured model or workflow with the input below.
            answer = model.invoke(f"Analyze {state['topic']} from the {angle} angle. Give one risk and one metric.")
            return {"findings": [f"{angle}: {answer.content}"]}
        return node

    builder = StateGraph(State)
    for angle in ("customer", "technical", "business", "safety"):
        builder.add_node(angle, analyze(angle))
        builder.add_edge(START, angle)
        builder.add_edge(angle, END)

    graph = builder.compile()
    result = graph.invoke({"topic": "AI support agent", "findings": []})
    print("\n\n".join(result["findings"]))


## Offline mechanics

This version runs without credentials and exposes the control flow directly.


In [ ]:
# Import the dependencies used by this example.
from concurrent.futures import ThreadPoolExecutor

# Define the data shape and small operations before running them.
def summarize_angle(angle):
    return f"{angle}: one risk, one opportunity, one metric"

angles = ["customer", "technical", "business", "safety"]
with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(summarize_angle, angles))

"\n".join(results)
